# Topic 10 — Practice: Pivot Tables & MultiIndex

**The notebook is the lesson.** Work top to bottom. Cells with `assert` grade themselves — a silent run = pass, an `AssertionError` = fix your code. Warm-Up, Interview Lens and Reflection have **no answer key** — answer in writing.

_Spaced repetition: ~60% of this notebook is the current topic, ~40% revisits earlier topics._

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
products = pd.read_csv(RAW+'products.csv', dtype={'product_id':str})
products['list_price'] = np.where(products['list_price']<0, np.nan, products['list_price'])
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str,'customer_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
orders['status'] = orders['status'].str.strip().str.lower()
orders['order_date'] = pd.to_datetime(orders['order_date'], format='mixed', dayfirst=True, errors='coerce')
orders['month'] = orders['order_date'].dt.to_period('M').astype(str)
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])
master = (items.merge(products[['product_id','category','unit_cost']], on='product_id', how='left')
              .merge(orders[['order_id','channel','status','month','order_date']], on='order_id', how='left'))
master['line_cogs'] = master['quantity']*master['unit_cost']
master['line_profit'] = master['line_revenue'] - master['line_cogs']


## 🔁 Warm-Up Recall (earlier topics — no answers given)

From **Topics 01–09**:

1. (T09) Order the toolbox fastest→slowest (vectorized…apply).
2. (T05) What does `transform` return vs `agg`?
3. NumPy: `np.reshape` changes shape — what's the pandas label-aware cousin?

In [ ]:
# NumPy recall: reshape a flat array into a 2-D grid
flat = np.arange(12)
grid = ...   # TODO 3x4
assert grid.shape == (3,4) and grid[1,2] == 6
print(grid)

## 🎯 Core Lesson Tasks (current topic)

`pivot_table` aggregates + reshapes (use it, not `pivot`, on real data). `unstack`/`stack` move levels.

**Core 1 — crosstab.** Row-normalised `crosstab` of `channel` × `status`. Which channel returns most?

In [ ]:
ct = ...
assert np.allclose(ct.sum(axis=1), 1)
ct.round(3)

**Core 2 — revenue pivot.** Revenue by `channel` (rows) × `month` (cols), `aggfunc='sum'`, with margins.

In [ ]:
pivot = ...
assert 'All' in pivot.index
pivot.iloc[:, :4]

**Core 3 — MultiIndex.** GroupBy `['category','channel']` revenue → MultiIndex Series; `unstack` channel to columns.

In [ ]:
grp = ...
assert isinstance(grp.index, pd.MultiIndex)
grp.unstack().round(0).head()

## 🔀 Mixed Review Tasks (earlier topics — spaced repetition)

**Mixed review (Topic 07) — month ordering.** Confirm `month` columns sort chronologically (string YYYY-MM works).

In [ ]:
months = sorted(master['month'].dropna().unique())
assert months == list(pd.Series(months).sort_values())
print(months[:3], '...', months[-1])

**Mixed review (Topic 06) — grain check.** Prove the master table didn't fan out: `len(master) == len(items)`.

In [ ]:
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
assert len(master) == len(items)
print('grain preserved:', len(master))

## 🕵️ Data Detective Investigation

**Case file #10 — the management cross-tab.** Produce the board-ready grid: revenue by `category × month`, then a heatmap. Which category drove the seasonal swing the totals hid?

In [ ]:
cat_month = master.pivot_table(index='category', columns='month', values='line_revenue', aggfunc='sum', fill_value=0)
assert cat_month.shape[1] >= 12
plt.figure(figsize=(12,4)); plt.imshow(cat_month, aspect='auto')
plt.yticks(range(len(cat_month)), cat_month.index); plt.title('Revenue: category x month'); plt.colorbar(); plt.show()
print(cat_month.sum(axis=1).sort_values(ascending=False).head())

## 🔎 Interview Lens (write answers — no model answers)

- **Q15:** When `groupby` over `pivot_table`, and vice versa?
- **Q17:** What do `observed=` and `dropna=` control on a groupby, and when do the defaults surprise you?

## ✍️ Reflection (write short explanations)

1. Answer Q15 in writing.
2. Which category drove the seasonal swing?
3. **Investigation log:** what does the cross-tab reveal that the totals hid?